**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `exploitations.csv`

## 1. Description du projet

Ce notebook a pour objectif de générer le fichier `exploitations.csv`, une des entrées requises par le module agricole de MAELIA.

En partant du parcellaire enrichi, qui contient des informations sur chaque parcelle, nous allons extraire une liste **unique** de tous les exploitants agricoles et leur assigner un identifiant standardisé ainsi qu'un type.

---
## 2. Objectifs

* Charger le fichier `parcellaire_enrichi.shp`.
* Extraire la liste unique des exploitants à partir de la colonne `'NOM_UTILIS'`.
* Créer un nouvel identifiant standardisé pour chaque exploitant (ex: `'SSM1-0001'`).
* Définir un type d'exploitant (`avec_UTL` / `sans_UTL`) en se basant sur la colonne `'UTL_2012'`.
* Sauvegarder le résultat (`ID_EXPL`, `TYPE_EXPL`) dans un fichier CSV.

---
## 3. Fichiers en Entrée et en Sortie

### 3.1. Fichier en Entrée
* **Parcellaire Enrichi :** `data/sols/shapefiles/processed/parcellaire_enrichi.shp`

### 3.2. Fichier en Sortie
* **Liste des Exploitations :** `includes_sassemeV1/modeleAgricole/agriculteurs/exploitations.csv`
---

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

In [2]:
# --- CHEMINS ---
base_dir = Path.cwd().parent.resolve()
input_parcellaire_path = base_dir / "data" / "sols" / "shapefiles" / "processed" / "parcellaire_enrichi.shp"
output_exploitations_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "agriculteurs" / "exploitations.csv"

In [3]:
try:
    gdf_parcellaire = gpd.read_file(input_parcellaire_path)
    print(f"✅ Parcellaire enrichi chargé ({len(gdf_parcellaire)} lignes).")
except Exception as e:
    print(f"🚨 ERREUR lors du chargement du fichier : {e}")

✅ Parcellaire enrichi chargé (749 lignes).


In [5]:
# --- IDENTIFICATION DES EXPLOITANTS UNIQUES ---
# On isole les colonnes nécessaires et on supprime les doublons basés sur le nom
df_exploitants = gdf_parcellaire[['NOM_UTILIS', 'UTL_2012']].copy()
df_exploitants.drop_duplicates(subset=['NOM_UTILIS'], inplace=True, keep='first')
df_exploitants.reset_index(drop=True, inplace=True) # On réinitialise l'index
print(f"-> {len(df_exploitants)} exploitants uniques trouvés.")

-> 44 exploitants uniques trouvés.


In [6]:
# --- NETTOYAGE ET CRÉATION DES COLONNES FINALES ---

# S'assurer que 'UTL_2012' est numérique
# On convertit en numérique, les erreurs deviendront NaN
df_exploitants['UTL_2012'] = pd.to_numeric(df_exploitants['UTL_2012'], errors='coerce')
# On remplace les NaN par 0 pour les calculs
df_exploitants['UTL_2012'].fillna(0, inplace=True)
print("-> Colonne 'UTL_2012' nettoyée et convertie en numérique.")

# 3b. Création de ID_EXPL
df_exploitants['ID_EXPL'] = 'SSM1-' + (df_exploitants.index + 1).astype(str).str.zfill(4)
print("-> Colonne 'ID_EXPL' créée.")

# 3c. Création de TYPE_EXPL
df_exploitants['TYPE_EXPL'] = np.where(df_exploitants['UTL_2012'] > 0, 'avec_UTL', 'sans_UTL')
print("-> Colonne 'TYPE_EXPL' créée.")

-> Colonne 'UTL_2012' nettoyée et convertie en numérique.
-> Colonne 'ID_EXPL' créée.
-> Colonne 'TYPE_EXPL' créée.


C:\Users\Cheikhou\AppData\Local\Temp\ipykernel_24432\2114513062.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_exploitants['UTL_2012'].fillna(0, inplace=True)


In [7]:
df_final = df_exploitants[['ID_EXPL', 'TYPE_EXPL']]
output_exploitations_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(output_exploitations_path, index=False, sep=';')
print(f"\n✅ Fichier 'exploitations.csv' sauvegardé avec succès.")

# --- VÉRIFICATION ---
print("\nAperçu du fichier final des exploitations :")
display(df_final.head())
print(f"\nRépartition des types d'exploitants :")
display(df_final['TYPE_EXPL'].value_counts())


✅ Fichier 'exploitations.csv' sauvegardé avec succès.

Aperçu du fichier final des exploitations :


,ID_EXPL,TYPE_EXPL
0,SSM1-0001,sans_UTL
1,SSM1-0002,sans_UTL
2,SSM1-0003,sans_UTL
3,SSM1-0004,sans_UTL
4,SSM1-0005,sans_UTL



Répartition des types d'exploitants :


TYPE_EXPL
sans_UTL    42
avec_UTL     2
Name: count, dtype: int64